# Tutorial 5: Deep Agents, Building a Harness and a Team of Agents

Tutorial 4 ended with `create_agent`: one call that builds the ReAct loop. That loop is enough for short tasks.
Long tasks (write a report, study a codebase, prepare an exam) break it in predictable ways: the model forgets the
plan, the context fills with raw tool output, and one agent tries to do everything at once.

**Deep Agents** (`pip install deepagents`) is LangChain's answer. It is not a new kind of agent. It is the same
`create_agent` loop wrapped in a **harness**: a planning tool, a filesystem, subagents, automatic summarization and
optional memory, skills, permissions and human approval.

By the end you will be able to:

- say what a **harness** is, and what it adds on top of the model and the ReAct loop;
- build a deep agent with **`create_deep_agent`** and read its virtual **filesystem** and **todo list**;
- customise the harness piece by piece: **tools, prompt, planning, middleware, permissions, human approval**;
- choose a **backend** (state, real disk, long-term store) for where the agent's files live;
- give an agent **memory** (`AGENTS.md`) and **skills** (`SKILL.md`);
- build a **team of agents** with subagents and the `task` tool, watch them work, and plug in any LangGraph agent;
- decide between tutorial 4's hand-built multi-agent graphs and a deep agent, with a reason.

## 0. Mental model: agent = model + harness

The model only maps messages to a reply. Everything else that makes it useful is the **harness**: the code
around the model that decides what it sees, which tools it has, and what happens to their output.

```text
                      ┌──────────────────── harness ────────────────────┐
                      │  system prompt      planning (write_todos)       │
user ──► messages ───►│  filesystem tools   subagents (task)             │──► answer
                      │  summarization      memory / skills              │     + files
                      │  permissions        human approval               │
                      │            ┌───────────────────────┐             │
                      │            │ model ⇄ tools  (ReAct) │             │
                      │            └───────────────────────┘             │
                      └──────────────────────────────────────────────────┘
```

Claude Code, Codex and Manus are all "deep" agents in this sense: a fairly ordinary loop with a well designed
harness. The four ideas that matter most:

| Idea | Problem it solves | Deep Agents piece |
|---|---|---|
| **Plan** | The model loses track of a long task | `write_todos` tool (planning middleware) |
| **Offload context to files** | Tool output floods the context window | `ls`, `read_file`, `write_file`, `edit_file`, `glob`, `grep` |
| **Isolate context in subagents** | One context mixes many subtasks | `task` tool and subagents |
| **Detailed instructions** | The model does not know your process | `system_prompt`, memory, skills |

In LangChain each piece is a **middleware**: an object that adds tools, edits the prompt, or wraps model and
tool calls. A deep agent is `create_agent` with a carefully chosen middleware stack, and building your own
harness means choosing that stack.

## 1. Setup

Same environment as tutorials 3 and 4, plus `deepagents`:

```bash
pip install "langchain>=1.0" langchain-groq python-dotenv langgraph deepagents
```

and the same `.env` beside this notebook:

```text
GROQ_API_KEY=your-key-here
```

This notebook was written against `deepagents` 0.7. The library moves quickly; if a name below fails to import,
check the version with `pip show deepagents`.

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()
assert os.getenv("GROQ_API_KEY"), "GROQ_API_KEY not found, see tutorial 3, section 1."

from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage

# Deep agents make many tool calls per task, so a stronger model helps.
# If you hit rate limits, switch to "openai/gpt-oss-20b" (the model used in tutorial 4).
model = ChatGroq(model="openai/gpt-oss-120b", temperature=0)

# Deep agents take more steps than a plain ReAct agent; give them room, but keep a cap.
LIMITS = {"recursion_limit": 60}


def show_trace(messages, width=160):
    """Print one line per message: who spoke, and either the tool calls or the text."""
    for m in messages:
        kind = getattr(m, "name", None) or type(m).__name__
        calls = getattr(m, "tool_calls", None)
        if calls:
            body = "→ " + ", ".join(f"{c['name']}({_short(c['args'])})" for c in calls)
        else:
            body = str(m.content).replace("\n", " ")
        print(f"{kind:>14}: {body[:width]}")


def _short(args, n=60):
    text = ", ".join(f"{k}={v!r}" for k, v in args.items())
    return text if len(text) <= n else text[:n] + "..."


def show_files(files):
    """Print the agent's virtual filesystem: path, then content."""
    if not files:
        print("(no files)")
    for path, data in files.items():
        print(f"===== {path} =====")
        print(data["content"])
        print()


def ask(agent, prompt, config=None, **state):
    """Send one user message (plus optional extra state such as files) and return the final state."""
    return agent.invoke(
        {"messages": [HumanMessage(prompt)], **state},
        config={**LIMITS, **(config or {})},
    )


print("ready")

### Code walkthrough: what is new

**`ChatGroq` and `HumanMessage`** are unchanged from tutorial 4. Deep Agents accepts any LangChain chat model
that supports tool calling, as an object (as here) or as a string such as `"openai:gpt-5.5"`.

**`show_trace`** is tutorial 4's helper with one change: it prints the *arguments* of each tool call, because in a
deep agent the interesting part is often which file was written or which subagent was asked what.

**`show_files`** prints the agent's filesystem. A deep agent returns its files next to its messages, so they are
part of the result you inspect.

## 2. Your first deep agent

`create_deep_agent` has the same shape as `create_agent`: a model, optional tools, and a system prompt. Start with
no tools of your own, to see what the harness provides.

In [ ]:
from deepagents import create_deep_agent

agent = create_deep_agent(
    model=model,
    system_prompt=(
        "You are a teaching assistant for an LLM course. "
        "Keep your work in files: write drafts to files and edit them, "
        "rather than writing long text in your replies."
    ),
)

result = ask(
    agent,
    "Write a short study note on the KV cache to /notes/kv_cache.md (what it stores, why it speeds "
    "up generation, what it costs in memory). Then add a one-line TL;DR at the top of that file.",
)

show_trace(result["messages"])
print()
show_files(result.get("files", {}))

### Code walkthrough: the filesystem is part of the state

**The result has a `files` key next to `messages`.** You did not write any file tool; the harness gave the agent
`ls`, `read_file`, `write_file`, `edit_file`, `delete`, `glob` and `grep`. By default the files live in the
graph state (a `StateBackend`), so nothing touches your disk. Section 5 changes that.

**Read the trace.** A typical run is `write_file`, then `edit_file` (or `read_file` then `edit_file`), then a short
final reply. The reply is short because the work is in the file. That is **context offloading**: a long draft
lives in a file, and the model only sees it when it reads it.

**`edit_file` does a string replacement**, not a full rewrite. For a long file this is the difference between
sending one line and sending the whole document again.

The built-in tools, for reference:

| Tool | What it does |
|---|---|
| `ls`, `glob`, `grep` | List, find by pattern, search contents |
| `read_file` | Read a file, with line numbers, optionally a range of lines |
| `write_file` | Create a file |
| `edit_file` | Replace an exact string inside a file |
| `delete` | Delete a file |
| `task` | Delegate a subtask to a subagent (section 7) |
| `execute` | Run shell commands, only with a sandbox backend (not used here) |

## 3. Anatomy of the harness: the middleware stack

`create_deep_agent` returns a compiled LangGraph graph, the same kind of object as tutorial 4's agents. Its nodes
show the harness: the `model` and `tools` nodes are tutorial 4's ReAct loop, and every other node is a middleware
hook that runs before or after them.

In [ ]:
for node in agent.get_graph().nodes:
    print(node)

### Code walkthrough: what each middleware adds

| Middleware | Adds | Why it matters |
|---|---|---|
| `FilesystemMiddleware` | The file tools and the `files` state | Offload context; share work between agents |
| `SubAgentMiddleware` | The `task` tool and a `general-purpose` subagent | Isolate context per subtask |
| `SummarizationMiddleware` | Summarises old messages when the context gets long | Long tasks do not overflow the window |
| `PatchToolCallsMiddleware` | Repairs tool calls left without a result | An interrupted run can resume cleanly |
| Prompt caching middleware | Cache markers for providers that support them | Cheaper repeated prompts (no-op on Groq) |
| `SkillsMiddleware`, `MemoryMiddleware` | Only if you pass `skills=` or `memory=` | Section 6 |
| `HumanInTheLoopMiddleware` | Only if you pass `interrupt_on=` | Section 4.5 |

The base stack is the part you get for free. **Building your own harness means adding to it**: your tools, your
prompt, and your own middleware, which `create_deep_agent(middleware=[...])` inserts after the base stack.
Your middleware applies to the **main agent only**; subagents (section 7) build their own stack.

Note what is *not* in the default stack in `deepagents` 0.7: the planning tool `write_todos`. You add it yourself
in section 4.2, which is a good first example of extending the harness.

## 4. Building your own harness, step by step

The running example is a **course assistant**: it can search the course notes and look up exam dates, and it
writes study material into files. Each subsection adds one piece to the harness.

### 4.1 Domain tools and a system prompt

Your tools are passed exactly as in tutorial 4, and they are **added** to the built-in ones, never replacing them.

In [ ]:
from langchain_core.tools import tool

COURSE_NOTES = {
    "tokenization": "Tokenizers split text into subword units. BPE merges frequent pairs; "
                    "WordPiece is used by BERT; SentencePiece works on raw bytes or characters.",
    "attention": "Self-attention computes softmax(QK^T / sqrt(d)) V. Each head learns a different "
                 "pattern; cost grows with the square of the sequence length.",
    "kv cache": "During generation the keys and values of past tokens are cached, so each new token "
                "costs one forward step instead of reprocessing the whole prefix. Memory grows "
                "linearly with sequence length, layers and heads.",
    "lora": "LoRA freezes the base weights and learns a low-rank update BA, so only a small "
            "fraction of parameters is trained.",
    "rag": "Retrieval-augmented generation retrieves relevant chunks and adds them to the prompt, "
           "so the model answers from sources instead of from memory alone.",
}
EXAM_DATES = {"llm": "2026-12-20", "nlp": "2026-12-27"}


@tool
def search_course_notes(topic: str) -> str:
    """Return the course's lecture notes on a topic, e.g. 'attention', 'kv cache', 'lora'."""
    topic = topic.lower().strip()
    hits = [text for key, text in COURSE_NOTES.items() if key in topic or topic in key]
    return "\n".join(hits) or f"No notes found for '{topic}'. Known topics: {', '.join(COURSE_NOTES)}"


@tool
def get_exam_date(course: str) -> str:
    """Return the final exam date for a course, given a short name such as 'llm'."""
    return EXAM_DATES.get(course.lower().strip(), "no exam on record")


COURSE_PROMPT = """You are a teaching assistant for an LLM course.

Rules:
- Base every fact on search_course_notes. Never invent course content.
- Put study material in files under /study/. Keep your chat replies short and point to the files.
- When you finish, list the files you created."""

course_tools = [search_course_notes, get_exam_date]

### 4.2 Planning: add the `write_todos` tool

Planning in a deep agent is surprisingly simple. The `write_todos` tool does nothing except store a list of
tasks in the state. It works because the model **writes its plan down and re-reads it** at every step, so the plan
stays in front of it however long the task gets. `TodoListMiddleware` provides the tool and a prompt explaining
how to use it.

In [ ]:
from langchain.agents.middleware import TodoListMiddleware

planner = create_deep_agent(
    model=model,
    tools=course_tools,
    system_prompt=COURSE_PROMPT,
    middleware=[TodoListMiddleware()],
)

result = ask(
    planner,
    "My LLM exam is coming. Make me a study pack: one file per topic for attention, KV cache and LoRA, "
    "and a /study/plan.md with a day-by-day schedule from today until the exam. Plan first.",
)

show_trace(result["messages"])
print("\nFinal todo list:")
for item in result.get("todos", []):
    print(f"  [{item['status']:>11}] {item['content']}")
print("\nFiles:", list(result.get("files", {})))

### Code walkthrough: a plan is just state

**`middleware=[TodoListMiddleware()]`** extends the stack from section 3. The result now has a `todos` key, and
each `write_todos` call in the trace *replaces* the whole list with an updated one: items move from `pending` to
`in_progress` to `completed`.

**The plan costs tokens and calls.** For a one-step question a plan is pure overhead, which is one reason it is
not in the default stack. Add planning when tasks have several steps that must not be forgotten.

### 4.3 Your own middleware: observe and limit

Middleware is how you change the harness's behaviour without touching the loop. Two common needs: **see** what
the agent does, and **stop** it from running away. The decorator `@wrap_tool_call` turns a function into a
middleware that wraps every tool call; `ModelCallLimitMiddleware` is a ready-made limit.

In [ ]:
import time
from langchain.agents.middleware import wrap_tool_call, ModelCallLimitMiddleware


@wrap_tool_call
def log_tool_calls(request, handler):
    """Print each tool call, run it, and print how long it took."""
    call = request.tool_call
    start = time.perf_counter()
    result = handler(request)                    # run the real tool
    print(f"  [tool] {call['name']:<20} {time.perf_counter() - start:6.3f}s  {_short(call['args'])}")
    return result


harnessed = create_deep_agent(
    model=model,
    tools=course_tools,
    system_prompt=COURSE_PROMPT,
    middleware=[
        TodoListMiddleware(),
        log_tool_calls,
        ModelCallLimitMiddleware(run_limit=15, exit_behavior="end"),
    ],
)

result = ask(harnessed, "Write /study/rag.md with what RAG is and when to use it. Also, when is the LLM exam?")
print()
print(result["messages"][-1].content)

### Code walkthrough: middleware wraps, it does not replace

**`handler(request)` runs the real tool.** Code before it runs before the tool, code after it runs after. You can
also change the request, skip the call and return your own `ToolMessage`, or retry on errors. The same pattern
exists for model calls (`@wrap_model_call`) and for simple hooks (`@before_model`, `@after_model`).

**`ModelCallLimitMiddleware(run_limit=15, exit_behavior="end")`** is the termination condition from tutorial 4,
section 3, packaged as middleware. `recursion_limit` counts graph steps and raises an error; this counts model
calls and ends the run gracefully.

LangChain ships more ready-made middleware you can drop into the same list, for example
`ToolCallLimitMiddleware`, `ModelRetryMiddleware`, `ModelFallbackMiddleware` and `PIIMiddleware`.

### 4.4 Giving the agent input files

The filesystem works in both directions: you can **seed** it with files when you invoke the agent. This is how you
hand an agent a long document without pasting it into the prompt. File values use the same format you saw in
`result.get("files", {})`; `create_file_data` builds one from a string.

In [ ]:
from deepagents.backends.utils import create_file_data

syllabus = {
    "/syllabus/week1.md": create_file_data("Week 1: Hugging Face ecosystem, tokenization (BPE, WordPiece)."),
    "/syllabus/week2.md": create_file_data("Week 2: Transformer internals, attention, KV cache, decoding."),
    "/syllabus/week3.md": create_file_data("Week 3: LangChain, LangGraph, tool-calling agents."),
}

result = ask(
    harnessed,
    "Read the syllabus files and write /study/week2_questions.md with five exam questions for week 2.",
    files=syllabus,
)
print()
print(result.get("files", {})["/study/week2_questions.md"]["content"] if "/study/week2_questions.md" in result.get("files", {})
      else list(result.get("files", {})))

### 4.5 Guardrails: permissions and human approval

An agent with a filesystem can also overwrite things it should not. The harness gives two controls:

- **`permissions`**: rules checked on every file tool call. `deny` refuses, `interrupt` asks a human.
- **`interrupt_on`**: pause before chosen tools run, for any tool, not just file tools.

Both live in the harness, so they hold even if the model ignores the prompt.

In [ ]:
from deepagents import FilesystemPermission

guarded = create_deep_agent(
    model=model,
    tools=course_tools,
    system_prompt=COURSE_PROMPT,
    permissions=[
        FilesystemPermission(operations=["write"], paths=["/syllabus/**"], mode="deny"),
    ],
)

result = ask(
    guarded,
    "Add a line 'Week 2 also covers speculative decoding.' to /syllabus/week2.md.",
    files=syllabus,
)
show_trace(result["messages"])
print("\nweek2 unchanged:", result.get("files", {})["/syllabus/week2.md"]["content"])

Now human approval. Pausing needs somewhere to keep the paused run, so the agent needs a **checkpointer** and every
call needs a `thread_id` (tutorial 4's memory section used the same two ingredients). The run stops with an
`__interrupt__` key; you inspect the proposed calls and resume with a decision.

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

reviewed = create_deep_agent(
    model=model,
    tools=course_tools,
    system_prompt=COURSE_PROMPT,
    interrupt_on={"write_file": True, "edit_file": True},   # pause before these tools
    checkpointer=InMemorySaver(),
)

thread = {"configurable": {"thread_id": "hitl-demo"}}
result = ask(reviewed, "Write /study/lora.md with a short explanation of LoRA.", config=thread)

while "__interrupt__" in result:
    request = result["__interrupt__"][0].value
    decisions = []
    for action in request["action_requests"]:
        print(f"Agent wants to call {action['name']} with:")
        print("   ", _short(action["args"], 200))
        # In an application you would ask the user here. We approve writes under /study/ only.
        if action["args"].get("file_path", "").startswith("/study/"):
            decisions.append({"type": "approve"})
        else:
            decisions.append({"type": "reject", "message": "Only write under /study/."})
    result = reviewed.invoke(Command(resume={"decisions": decisions}), config={**LIMITS, **thread})

print("\nFinal reply:", result["messages"][-1].content)
print("Files:", list(result.get("files", {})))

### Code walkthrough: pause, inspect, resume

**`result["__interrupt__"][0].value["action_requests"]`** lists every gated tool call the model made in that
turn, with its arguments. Nothing has run yet.

**`Command(resume={"decisions": [...]})`** continues the same thread, with one decision per request, in order:
`approve`, `reject` (with a message the model will read), `edit` (run it with changed arguments) or `respond`.

**Why a loop?** After one approval the agent may want to write again, so a robust caller keeps answering until
there is no `__interrupt__`. If the model chose not to call a gated tool at all, the loop never runs.

## 5. Where do the files live? Backends

The file tools talk to a **backend**. Changing the backend changes where files are stored without changing the
agent or its tools:

| Backend | Files live in | Lifetime |
|---|---|---|
| `StateBackend` (default) | The graph state | One thread (a conversation) |
| `FilesystemBackend(root_dir=...)` | A real folder on disk | Until you delete it |
| `StoreBackend(store=...)` | A LangGraph store (memory, Postgres, ...) | Across threads |
| `CompositeBackend(default, routes)` | Different backends per path prefix | Mixed |

### 5.1 Real files on disk

`FilesystemBackend` maps the agent's `/` to a folder. With `virtual_mode=True` the agent cannot escape that folder,
for example with `..` paths. Treat it as giving the agent write access to that folder, nothing more.

In [ ]:
from pathlib import Path
from deepagents.backends import FilesystemBackend

WORKSPACE = Path("deep_agent_workspace").resolve()
WORKSPACE.mkdir(exist_ok=True)

disk_agent = create_deep_agent(
    model=model,
    tools=course_tools,
    system_prompt=COURSE_PROMPT,
    backend=FilesystemBackend(root_dir=WORKSPACE, virtual_mode=True),
)

result = ask(disk_agent, "Write /study/tokenization.md summarising the course notes on tokenization.")
print(result["messages"][-1].content)
print("\nOn disk:", [str(p.relative_to(WORKSPACE)) for p in WORKSPACE.rglob("*") if p.is_file()])

### 5.2 Long-term memory with a composite backend

A common layout: scratch files are per conversation, but anything under `/memories/` survives across
conversations. `CompositeBackend` routes by path prefix, and `StoreBackend` keeps its files in a LangGraph store.
We run two different threads; the second can read what the first saved under `/memories/`, but not its scratch
files.

In [ ]:
from deepagents.backends import CompositeBackend, StateBackend, StoreBackend
from langgraph.store.memory import InMemoryStore

store = InMemoryStore()   # use a database-backed store in production

memory_agent = create_deep_agent(
    model=model,
    system_prompt=(
        "You are a study coach. Save lasting facts about the student to /memories/student.md "
        "and check that file at the start of every conversation. Use other paths for scratch work."
    ),
    backend=CompositeBackend(
        default=StateBackend(),
        routes={"/memories/": StoreBackend(store=store, namespace=lambda runtime: ("student-42",))},
    ),
    store=store,
    checkpointer=InMemorySaver(),
)

r1 = ask(memory_agent,
         "Hi, I'm Sara. I prefer short bullet-point answers and my weak topic is attention. "
         "Also jot today's to-do in /scratch/today.md.",
         config={"configurable": {"thread_id": "monday"}})
print("Monday files in state:", list(r1.get("files", {})))

r2 = ask(memory_agent, "What do you know about me, and what was on my to-do list?",
         config={"configurable": {"thread_id": "tuesday"}})
print("\nTuesday reply:\n", r2["messages"][-1].content)
print("\nIn the store:", [item.key for item in store.search(("student-42",))])

### Code walkthrough: one agent, two lifetimes

**`routes={"/memories/": StoreBackend(...)}`** sends every path starting with `/memories/` to the store, and all
other paths to `StateBackend`. The agent sees one filesystem and does not know about the split.

**Threads separate state, not the store.** On Tuesday the scratch file is gone (it lived in Monday's state), but
`/memories/student.md` is still there. Notice that Monday's `files` key does not list `/memories/student.md`:
only state-backed files appear there.

**`namespace=lambda runtime: ("student-42",)`** decides whose memory this is. In an application you would build
the namespace from the logged-in user, so students never see each other's memories.

## 6. Memory files and skills

Two more harness features load instructions **from files**, so you can change behaviour without changing code:

- **Memory** (`memory=["/AGENTS.md"]`): the file is read at startup and **always** placed in the system prompt.
  Use it for standing facts and conventions. Claude Code's `CLAUDE.md` is the same idea.
- **Skills** (`skills=["/skills/"]`): each skill is a folder with a `SKILL.md`. Only the skill's **name and
  description** go in the prompt; the agent reads the full instructions with `read_file` when a task needs them.
  This is **progressive disclosure**: many skills cost almost nothing until one is used.

In [ ]:
(WORKSPACE / "AGENTS.md").write_text(
    "# Course assistant memory\n"
    "- Students are master's students; assume Python and linear algebra.\n"
    "- Always cite the lecture topic a fact comes from, e.g. (lecture: attention).\n",
    encoding="utf-8",
)

skill_dir = WORKSPACE / "skills" / "quiz-maker"
skill_dir.mkdir(parents=True, exist_ok=True)
(skill_dir / "SKILL.md").write_text(
    "---\n"
    "name: quiz-maker\n"
    "description: Write multiple-choice quiz questions with an answer key. Use when asked for a quiz.\n"
    "---\n\n"
    "# Quiz maker\n\n"
    "1. Look up each topic with search_course_notes before writing questions.\n"
    "2. Every question has exactly four options, labelled A to D, and one correct answer.\n"
    "3. Write the questions to /quizzes/<topic>.md and the answer key, with a one-line reason per answer, "
    "to /quizzes/<topic>_answers.md.\n",
    encoding="utf-8",
)

skilled = create_deep_agent(
    model=model,
    tools=course_tools,
    system_prompt=COURSE_PROMPT,
    backend=FilesystemBackend(root_dir=WORKSPACE, virtual_mode=True),
    memory=["/AGENTS.md"],
    skills=["/skills/"],
)

result = ask(skilled, "Make me a 3-question quiz on the KV cache.")
show_trace(result["messages"])

### Code walkthrough: what to look for in the trace

**A `read_file` call on `/skills/quiz-maker/SKILL.md`** early in the run. The agent saw only the description,
decided the skill applied, and loaded the full instructions itself.

**Files under `/quizzes/`** in the format the skill demands, which you can open in `deep_agent_workspace/`.

**Paths in `memory=` and `skills=` are backend paths**, so with `FilesystemBackend` they are relative to
`root_dir`. With the default `StateBackend` you would pass them as seeded files, as in section 4.4.

| Put it in... | When |
|---|---|
| `system_prompt` | The agent's role and rules that never change |
| Memory (`AGENTS.md`) | Facts and conventions that every task needs, edited by people (or by the agent) |
| A skill (`SKILL.md`) | A procedure needed only for some tasks |
| A tool | Something the agent must *do*, not just know |

## 7. Multiple agents: subagents and the `task` tool

Tutorial 4, section 6.2 built a supervisor by hand: each worker was an agent wrapped in a `@tool` function. Deep
Agents makes this the default. The **`task` tool** takes a subagent name and a task description, runs that
subagent in a **fresh context**, and returns only its final answer:

```text
main agent ──task("research attention", researcher)──► researcher  (own context, own tools)
     ▲                                                    │ many tool calls, long outputs
     └────────────────── final answer only ◄──────────────┘
```

This is **context quarantine**. The subagent's many intermediate steps never enter the main agent's context, so
the main agent stays focused on coordinating. The cost, as in tutorial 4, is more model calls.

### 7.1 The built-in `general-purpose` subagent

Every deep agent already has one subagent, `general-purpose`. It gets the main agent's model, your tools, the file
tools, skills and permissions, but a short generic prompt of its own and **none of your custom middleware** (no
`write_todos`, no `log_tool_calls`). The main agent can use it to split off a self-contained piece of work.

In [ ]:
result = ask(
    harnessed,
    "For each of tokenization, attention and LoRA, use a separate subagent to write a study note to "
    "/study/<topic>.md. Then write /study/index.md linking the three notes.",
)
show_trace(result["messages"])
print("\nFiles:", list(result.get("files", {})))

### Code walkthrough: what the main agent saw

**Count the messages.** The main trace shows `task(...)` calls and their short results, not the subagents'
`search_course_notes` and `write_file` calls. Even the `[tool]` lines from `log_tool_calls` show only the main
agent's calls (`task`, and its own file work), because custom middleware is not passed down to subagents. To see
inside a subagent you stream it (section 7.3) or give the subagent spec its own `middleware`.

**Files are shared.** Subagents use the same backend, so the notes they wrote appear in the main agent's `files`.
The filesystem is how agents **pass large results** to each other: a subagent writes a file and returns a one-line
summary with its path, instead of pasting the whole content back.

**Several `task` calls in one message run in parallel**, which is why the three notes can be written at the same
time.

### 7.2 A team of specialised subagents

A custom subagent is a dictionary: a `name`, a `description` that tells the main agent **when** to use it, a
`system_prompt`, and optionally its own `tools` and `model`. Here is a small team for writing a study guide:

| Agent | Tools | Job |
|---|---|---|
| Main (coordinator) | file tools, `task`, `write_todos` | Plan, delegate, assemble `/guide.md` |
| `researcher` | `search_course_notes`, file tools | Collect facts on one topic into `/research/<topic>.md` |
| `quiz-writer` | file tools | Write questions from the research files |
| `critic` | file tools (read only) | Check the guide against the research, report problems |

In [ ]:
researcher = {
    "name": "researcher",
    "description": "Collects the course facts on ONE topic and saves them to /research/<topic>.md. "
                   "Give it exactly one topic per call.",
    "system_prompt": "You are a researcher. Use search_course_notes for the topic you are given. "
                     "Write the facts as bullet points to /research/<topic>.md. "
                     "Reply with only the file path and a one-sentence summary.",
    "tools": [search_course_notes],
    # "model": ChatGroq(model="openai/gpt-oss-20b"),   # a cheaper model is fine for simple subagents
}

quiz_writer = {
    "name": "quiz-writer",
    "description": "Writes 2 exam questions per topic from the files in /research/. Call after research is done.",
    "system_prompt": "You write exam questions. Read the files in /research/ (use ls and read_file). "
                     "Write 2 questions per topic, with answers, to /quiz.md. Use only facts from the files. "
                     "Reply with only the file path.",
    "tools": [],
}

critic = {
    "name": "critic",
    "description": "Reviews /guide.md against the files in /research/ and reports factual errors or gaps.",
    "system_prompt": "You are a strict reviewer. Read /guide.md and every file in /research/. "
                     "List each claim in the guide that the research does not support, and each research fact "
                     "the guide misses. If there are none, reply exactly 'APPROVED'. Do not edit any file.",
    "tools": [],
    # The critic may read but not write: its own rules replace the main agent's.
    "permissions": [FilesystemPermission(operations=["write"], paths=["/**"], mode="deny")],
}

TEAM_PROMPT = """You coordinate a team that writes study guides for an LLM course.

Process:
1. Make a plan with write_todos.
2. Call the researcher once per topic. Call them in parallel.
3. Read the research files and write /guide.md: one section per topic, plain explanations.
4. Call the quiz-writer.
5. Call the critic. If it reports problems, fix /guide.md with edit_file and call the critic again (at most twice).
6. Reply with the list of files you produced.

Do not do the subagents' work yourself."""

team = create_deep_agent(
    model=model,
    system_prompt=TEAM_PROMPT,
    subagents=[researcher, quiz_writer, critic],
    middleware=[TodoListMiddleware(), ModelCallLimitMiddleware(run_limit=25, exit_behavior="end")],
)

result = ask(team, "Write a study guide on attention, the KV cache and RAG.")
show_trace(result["messages"])
print("\nFiles:", list(result.get("files", {})))

In [ ]:
show_files({p: d for p, d in result.get("files", {}).items() if p in ("/guide.md", "/quiz.md")})

### Code walkthrough: designing a team

**The `description` is routing logic**, exactly like a tool docstring in tutorial 4. The main agent chooses a
subagent from its name and description only. "Give it exactly one topic per call" is what makes the coordinator
fan out to parallel calls.

**Subagents start from nothing.** A subagent sees only the `description` argument of its `task` call, not the
conversation. Anything it needs must be in that string or in a file. This is why the team communicates through
`/research/` files and short replies.

**`tools` gives each agent only what it needs.** The quiz-writer cannot call `search_course_notes`, so it cannot
add facts that the researcher did not record. The critic also gets its own `permissions`, which replace the
inherited ones, so it can read everything and write nothing.

**The process lives in the prompt; the limits live in the harness.** "At most twice" in the prompt is a request.
`ModelCallLimitMiddleware` is a guarantee.

### 7.3 Watching the team work

`invoke` only returns the main agent's messages. To see the subagents too, **stream** with `subgraphs=True`. Each
update comes with a **namespace**: empty for the main agent, non-empty inside a subagent. Subagent messages also
carry the subagent's `name`.

In [ ]:
def watch(agent, prompt):
    stream = agent.stream(
        {"messages": [HumanMessage(prompt)]},
        config=LIMITS,
        stream_mode="updates",
        subgraphs=True,
    )
    for namespace, update in stream:
        for node, data in update.items():
            if not isinstance(data, dict) or not data.get("messages"):
                continue
            messages = data["messages"]
            messages = getattr(messages, "value", messages)        # unwrap deferred updates if any
            for m in messages if isinstance(messages, list) else [messages]:
                who = "main" if not namespace else (getattr(m, "name", None) or "subagent")
                indent = "" if not namespace else "    "
                calls = getattr(m, "tool_calls", None)
                body = ", ".join(f"{c['name']}({_short(c['args'], 50)})" for c in calls) if calls \
                    else str(m.content).replace("\n", " ")[:110]
                print(f"{indent}{who:>12} | {type(m).__name__:<12} | {body}")


watch(team, "Write a study guide on tokenization and LoRA.")

### Code walkthrough: reading the stream

**Indented lines are inside a subagent.** You can now see each researcher's `search_course_notes` and
`write_file` calls, which `invoke` hid. Two researchers running in parallel interleave their lines.

**`stream_mode="updates"`** yields what each node added, as it happens. In an application this is what you would
send to the user interface so they can see progress during a long run.

### 7.4 Plugging in any agent: `CompiledSubAgent`

A subagent does not have to be a dictionary. Any compiled LangGraph graph with a `messages` key can join the team,
including the `create_agent` agents and hand-built graphs from tutorial 4. Here the critic is replaced by a
`create_agent` with **structured output**, so the coordinator receives a validated JSON verdict instead of free
text (the same `Review` idea as tutorial 4, section 6.1).

In [ ]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from deepagents import CompiledSubAgent


class Review(BaseModel):
    approved: bool = Field(description="True if every claim in the text is supported by the course notes.")
    problems: list[str] = Field(description="Unsupported or wrong claims; empty if approved.")


fact_checker_graph = create_agent(
    model,
    tools=[search_course_notes],
    system_prompt="You fact-check text about the LLM course. Look up each topic with search_course_notes "
                  "and compare. The text to check is in the task you are given.",
    response_format=ToolStrategy(Review),
    name="fact-checker",
)

fact_checker = CompiledSubAgent(
    name="fact-checker",
    description="Fact-checks a short text against the course notes. Put the FULL text to check in the task.",
    runnable=fact_checker_graph,
)

checked_team = create_deep_agent(
    model=model,
    tools=[search_course_notes],
    system_prompt="Answer the student's question in one paragraph, then send that paragraph to the "
                  "fact-checker. If it is not approved, fix the paragraph. Reply with the final paragraph.",
    subagents=[fact_checker],
)

result = ask(checked_team, "Explain why LoRA is cheaper to train than full fine-tuning.")
show_trace(result["messages"])

### Code walkthrough: the subagent boundary

**`CompiledSubAgent(name, description, runnable)`** wraps an existing graph. The harness does not add file tools,
middleware or permissions to it: you get exactly the graph you built. A dictionary subagent, by contrast, is built
by the harness and gets its file tools automatically.

**Structured output crosses the boundary as JSON.** When the runnable returns a `structured_response`, the `task`
tool result is that object serialised as JSON (look for `{"approved": ...}` in the trace). Otherwise it is the
subagent's last message.

**When to use which:** a dictionary for an ordinary helper; `CompiledSubAgent` when the helper needs a custom graph,
structured output, a different state, or already exists in your code.

## 8. Choosing between them

| | Tutorial 4: `create_agent` and hand-built graphs | Deep Agents: `create_deep_agent` |
|---|---|---|
| Who decides the steps | You (pipeline) or a model (supervisor) | The model, guided by prompt and plan |
| Multi-agent wiring | You write nodes, edges or `@tool` wrappers | Subagent specs and the built-in `task` tool |
| Passing large results | Through state keys you design | Through a shared filesystem |
| Long tasks | You add memory and limits yourself | Planning, summarization and limits as middleware |
| Predictability | High: control flow is code | Lower: control flow is the model's choice |
| Cost per request | Low | Higher: more calls, longer prompts |

**Use a plain `create_agent`** for short tool-using tasks: a question needing a few lookups.

**Use a hand-built graph** when you know the steps and must guarantee them, for example a fixed review process
that must always run.

**Use a deep agent** when the task is long and open-ended, needs intermediate artefacts (drafts, notes, code), or
splits into independent subtasks. Remember that a deep agent is still `create_agent` underneath: everything you can
add to one, you can add to the other.

## Glossary: quick reference

| Term | Meaning |
|---|---|
| **Harness** | Everything around the model: prompt, tools, loop, middleware, storage, limits. |
| **Deep agent** | A `create_agent` loop with a harness for planning, files, subagents and summarization. |
| **`create_deep_agent`** | Builds a deep agent; returns a compiled LangGraph graph. |
| **Middleware** | An object that adds tools or state, edits the prompt, or wraps model and tool calls. |
| **`write_todos`** | Planning tool from `TodoListMiddleware`; stores the plan in `todos`. |
| **Context offloading** | Keeping large content in files and reading it only when needed. |
| **Backend** | Where the file tools store files: state, disk, store, or a composite. |
| **`task` tool** | Runs a subagent on a description in a fresh context and returns its final answer. |
| **Subagent** | A helper agent the main agent delegates to; a dict spec or a `CompiledSubAgent`. |
| **Context quarantine** | Keeping a subtask's intermediate steps out of the main agent's context. |
| **Memory (`AGENTS.md`)** | A file always loaded into the system prompt. |
| **Skill (`SKILL.md`)** | Instructions loaded on demand; only name and description are always visible. |
| **Permissions** | Allow, deny or interrupt rules on file paths, enforced by the harness. |
| **`interrupt_on`** | Pause before chosen tools for human approval; needs a checkpointer and `thread_id`. |